# Polaris: Gemma 4 Decision Lab

### Executed companion notebook for the live Polaris application

**Team:** Imtiaz Hossain and Mofftasim Hossain Sayem  
**Track:** Open Innovation  
**Live demo:** https://polaris-gemma4.vercel.app/demo  
**Repository:** https://github.com/ImtiazHossain-Eshan/polaris-gemma4

Polaris is a decision system, not a generic writing assistant. It turns a student's profile, constraints, evidence, deadlines, and available time into a measurable roadmap. This notebook exposes the model-facing workflows used by the latest web application.


## What this notebook proves

1. **Gemma 4 is the only generative model.** The model identifier is fixed and validated.
2. **Gemma 4 is central to the product.** It reasons about decisions, audits evidence, reviews exam performance, parses routines, and produces Bengali guidance.
3. **Retrieval and scoring remain inspectable.** Deterministic systems provide evidence and measurements without replacing the model.
4. **Structured outputs are validated.** The product checks every model response before it reaches the interface.
5. **The public demo is reproducible.** The same contracts shown here power Decision Twin, Evidence Graph, Mock Exams, and Smart Routine.

```mermaid
flowchart LR
  A[Student profile and request] --> B[Validation]
  B --> C[Deterministic evidence retrieval]
  C --> D[Gemma 4 reasoning]
  D --> E[Schema validation]
  E --> F[Roadmap and Action Lab]
  E --> G[Strategist response]
  E --> H[Routine and exam review]
```


## 1. Environment

On Kaggle, add a private secret named `GEMMA_API_KEY`. The secret is never printed or saved in notebook output.


In [1]:
# Kaggle already provides Python. Uncomment only if the SDK is unavailable.
# !pip install -q google-genai

import json
import os
import re
from collections import Counter
from math import log

from google import genai
from google.genai import types

def read_secret(name: str) -> str:
    value = os.environ.get(name, "")
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""

API_KEY = read_secret("GEMMA_API_KEY")
MODEL = os.environ.get("GEMMA_MODEL", "gemma-4-26b-a4b-it")
ALLOWED_MODELS = {"gemma-4-26b-a4b-it", "gemma-4-31b-it"}
assert API_KEY, "Add GEMMA_API_KEY through Kaggle Secrets before running."
assert MODEL in ALLOWED_MODELS

client = genai.Client(api_key=API_KEY)
print("Gemma client ready")
print("Model:", MODEL)
print("Credential present:", bool(API_KEY), "(value hidden)")


Gemma client ready
Model: gemma-4-26b-a4b-it
Credential present: True (value hidden)


## 2. Bangladesh-context student and evidence base

The sample is a Bangladeshi HSC student targeting competitive Computer Science programs with a score gap, limited weekly time, and a funding constraint.


In [2]:
student = {
    "country": "Bangladesh",
    "stage": "HSC / Class 12",
    "target_degree": "Computer Science undergraduate",
    "target_countries": ["United States", "Canada"],
    "gpa": 3.80,
    "sat": 1320,
    "sat_target": 1500,
    "ielts": 6.5,
    "weekly_hours": 14,
    "budget_bdt": 180_000,
    "strengths": ["one deployed student portal", "school club leadership"],
    "gaps": ["testing", "research evidence", "measured project impact"],
}

knowledge_base = [
    {"id": "testing", "title": "Testing evidence", "text": "Use timed diagnostics, keep an error log by skill, and retest after a focused practice cycle."},
    {"id": "projects", "title": "Project evidence", "text": "Verify shipped work through public artifacts, repository history, user adoption, references, and measured outcomes."},
    {"id": "funding", "title": "Funding feasibility", "text": "Track aid eligibility, total cost, required essays, and scholarship deadlines for every university."},
    {"id": "bangladesh", "title": "Bangladesh execution context", "text": "Protect HSC performance while preparing for tests and account for school hours, BDT budgets, and mentor access."},
]
print(json.dumps(student, indent=2, ensure_ascii=False))


{
  "country": "Bangladesh",
  "stage": "HSC / Class 12",
  "target_degree": "Computer Science undergraduate",
  "target_countries": [
    "United States",
    "Canada"
  ],
  "gpa": 3.8,
  "sat": 1320,
  "sat_target": 1500,
  "ielts": 6.5,
  "weekly_hours": 14,
  "budget_bdt": 180000,
  "strengths": [
    "one deployed student portal",
    "school club leadership"
  ],
  "gaps": [
    "testing",
    "research evidence",
    "measured project impact"
  ]
}


## 3. Inspectable retrieval

Polaris ranks evidence deterministically. Gemma 4 receives only the most relevant records, keeping the prompt compact and the retrieval trace visible.


In [3]:
def tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def retrieve(query, documents, k=3):
    query_terms = tokens(query)
    document_terms = [tokens(d["title"] + " " + d["text"]) for d in documents]
    n = len(documents)
    frequency = Counter(term for terms in document_terms for term in set(terms))
    rows = []
    for document, terms in zip(documents, document_terms):
        counts = Counter(terms)
        score = sum(counts[term] * log((n + 1) / (frequency[term] + 0.5)) for term in query_terms if counts[term])
        rows.append((score, document))
    return [document for _, document in sorted(rows, key=lambda row: row[0], reverse=True)[:k]]

query = "SAT moved earlier, protect HSC, prove project impact, limited BDT budget"
evidence = retrieve(query, knowledge_base)
for rank, item in enumerate(evidence, 1):
    print(f"{rank}. {item['title']} ({item['id']})")


1. Bangladesh execution context (bangladesh)
2. Project evidence (projects)
3. Testing evidence (testing)


## 4. Structured Gemma 4 helper

The live application uses shallow JSON contracts so the output can be validated, rendered safely, and retried when a required field is missing.


In [4]:
def gemma_json(system_instruction, prompt, schema, max_tokens=700):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.2,
            max_output_tokens=max(max_tokens, 1800),
            response_mime_type="application/json",
            response_schema=schema,
        ),
    )
    if response.parsed is not None:
        return response.parsed
    if response.text:
        return json.loads(response.text)
    raise RuntimeError("Gemma returned no structured payload")

print("Structured Gemma helper ready")


Structured Gemma helper ready


## 5. Decision Twin

A changed constraint is compared with the complete student context. Gemma 4 selects what should move, explains the trade-off, and names the first measurable action.


In [5]:
decision_schema = {
    "type": "object",
    "properties": {
        "summary": {"type": "string"},
        "focus": {"type": "string"},
        "next_action": {"type": "string"},
        "evidence": {"type": "string"},
    },
    "required": ["summary", "focus", "next_action", "evidence"],
}
scenario = "The SAT test date moved six weeks earlier."
decision = gemma_json(
    "You are the compact decision engine inside Polaris. Never promise admission. Gemma 4 is the only generative model.",
    f"STUDENT:
{json.dumps(student)}
CHANGE:
{scenario}
EVIDENCE:
{json.dumps(evidence)}
Keep every field under 30 words.",
    decision_schema,
)
print(json.dumps(decision, indent=2, ensure_ascii=False))


{
  "summary": "The compressed SAT timeline requires an immediate shift toward high-frequency testing preparation without sacrificing HSC performance.",
  "focus": "Accelerated SAT preparation, weekly diagnostics, and protected HSC study blocks.",
  "next_action": "Complete a timed SAT diagnostic within 24 hours and tag every mistake by skill.",
  "evidence": "Diagnostic score, categorized error log, and seven days of completed targeted practice."
}


## 6. Evidence-to-Action Graph

A student claim becomes useful only when a reviewer can verify it. Gemma 4 maps the claim through supplied proof, readable signal, remaining gap, next action, and verification.


In [6]:
evidence_schema = {
    "type": "object",
    "properties": {
        "signal": {"type": "string"},
        "gap": {"type": "string"},
        "next_action": {"type": "string"},
        "verification": {"type": "string"},
    },
    "required": ["signal", "gap", "next_action", "verification"],
}
claim = "I built a student portal used by 120 learners."
proof = "Public repository, deployment analytics, and two teacher references."
evidence_graph = gemma_json(
    "You are the evidence auditor inside Polaris. Do not verify unsupported claims.",
    f"CLAIM: {claim}
PROOF: {proof}",
    evidence_schema,
)
print(json.dumps({"claim": claim, "proof": proof, **evidence_graph}, indent=2))


{
  "claim": "I built a student portal used by 120 learners.",
  "proof": "Public repository, deployment analytics, and two teacher references.",
  "signal": "Deployment analytics and repository history support a real, shipped student product.",
  "gap": "The usage total still needs an independently dated source and an engagement metric.",
  "next_action": "Export a dated analytics snapshot and ask one teacher to confirm active learner usage.",
  "verification": "Cross-check the analytics date, repository release, and signed teacher confirmation."
}


## 7. Smart Routine

Gemma 4 converts a natural-language request into a strict weekly schedule block. The UI shows the parsed block for manual confirmation and editing before it is saved.


In [7]:
routine_schema = {
    "type": "object",
    "properties": {
        "day": {"type": "string", "enum": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]},
        "start": {"type": "string"},
        "end": {"type": "string"},
        "title": {"type": "string"},
        "category": {"type": "string", "enum": ["study", "exam", "project", "wellbeing", "application"]},
    },
    "required": ["day", "start", "end", "title", "category"],
}
routine = gemma_json(
    "Convert one request into one editable weekly schedule block. Use 24-hour HH:MM time.",
    "Add SAT math practice on Thursday from 9 to 10 pm.",
    routine_schema,
    300,
)
print(json.dumps({**routine, "source": "gemma4", "model": MODEL}, indent=2))


{
  "day": "Thursday",
  "start": "21:00",
  "end": "22:00",
  "title": "SAT Math Practice",
  "category": "study",
  "source": "gemma4",
  "model": "gemma-4-26b-a4b-it"
}


## 8. Adaptive mock-exam review

Question scoring is deterministic and auditable. Gemma 4 receives only the score and weak skills, then returns a concise recovery plan. Formula notation remains normal Markdown and LaTeX-safe in the application renderer.


In [8]:
exam_prompt = "SAT practice: 3/5. Weak skills: linear equations, transitions. Give a three-step prescription under 80 words."
exam_feedback = client.models.generate_content(
    model=MODEL,
    contents=exam_prompt,
    config=types.GenerateContentConfig(
        system_instruction="You are a precise exam coach. This is unofficial practice. Gemma 4 is the only generative model used.",
        temperature=0.2,
        max_output_tokens=220,
    ),
).text
print(exam_feedback)


1. Master linear equations by isolating variables and interpreting slope-intercept form, y = mx + b.
2. Group transition words by addition, contrast, and causation so sentence relationships are explicit.
3. Complete one untimed targeted set, review every error, then repeat the same skills under time pressure.


## 9. Bengali reasoning

The product can request natural Bengali reasoning while preserving genuine proper names and admissions acronyms such as SAT, IELTS, GPA, and HSC.


In [9]:
bengali_schema = {
    "type": "object",
    "properties": {
        "diagnosis": {"type": "string"},
        "today": {"type": "string"},
        "metric": {"type": "string"},
    },
    "required": ["diagnosis", "today", "metric"],
}
bengali = gemma_json(
    "আপনি Polaris-এর ভর্তি কৌশলবিদ। স্বাভাবিক বাংলায় উত্তর দিন। SAT, IELTS, GPA ও HSC অপরিবর্তিত রাখুন।",
    "SAT পরীক্ষার তারিখ ছয় সপ্তাহ এগিয়ে এসেছে। একটি সংক্ষিপ্ত বিশ্লেষণ, আজকের কাজ ও পরিমাপযোগ্য ফলাফল দিন।",
    bengali_schema,
    450,
)
print(json.dumps(bengali, indent=2, ensure_ascii=False))


{
  "diagnosis": "SAT প্রস্তুতির সময় কমে যাওয়ায় এখন দ্রুত অনুশীলন দরকার, তবে HSC-এর নিয়মিত পড়াশোনা বাদ দেওয়া যাবে না।",
  "today": "আজ ৪৫ মিনিটের একটি সময়বদ্ধ SAT পরীক্ষা দিন এবং ভুলগুলো দক্ষতা অনুযায়ী লিখে রাখুন।",
  "metric": "সাত দিনের মধ্যে তিনটি লক্ষ্যভিত্তিক অনুশীলন সম্পন্ন করুন এবং পরবর্তী পরীক্ষায় ভুলের সংখ্যা তুলনা করুন।"
}


## 10. Automated engineering checks

These checks validate the contracts that the interface depends on: required fields, measurable actions, valid schedule time, model trace, and Bengali-script coverage.


In [10]:
def has_all(obj, fields):
    return all(isinstance(obj.get(field), str) and obj[field].strip() for field in fields)

evaluation = {
    "decision_schema_valid": has_all(decision, ["summary", "focus", "next_action", "evidence"]),
    "evidence_schema_valid": has_all(evidence_graph, ["signal", "gap", "next_action", "verification"]),
    "routine_schema_valid": has_all(routine, ["day", "start", "end", "title", "category"]),
    "routine_time_valid": bool(re.fullmatch(r"[0-2]\d:[0-5]\d", routine["start"])) and routine["start"] < routine["end"],
    "decision_mentions_test": bool(re.search(r"SAT|test|diagnostic|score", json.dumps(decision), re.I)),
    "evidence_is_measurable": bool(re.search(r"score|log|analytics|date|week|%", decision["evidence"], re.I)),
    "bengali_script_present": bool(re.search(r"[\u0980-\u09FF]", json.dumps(bengali, ensure_ascii=False))),
    "only_allowlisted_model": MODEL in ALLOWED_MODELS,
}
for name, value in evaluation.items():
    print(f"{'PASS' if value else 'CHECK'}  {name}")
print(f"\nEngineering checks: {sum(evaluation.values())}/{len(evaluation)} passed")


PASS  decision_schema_valid
PASS  evidence_schema_valid
PASS  routine_schema_valid
PASS  routine_time_valid
PASS  decision_mentions_test
PASS  evidence_is_measurable
PASS  bengali_script_present
PASS  only_allowlisted_model

Engineering checks: 8/8 passed


## 11. How this maps to the latest live prototype

| Notebook proof | Live Polaris experience |
|---|---|
| Deterministic retrieval trace | Source-aware Strategist |
| Structured Decision Twin JSON | Interactive before-and-after roadmap diff |
| Evidence audit | Claim to proof to signal to gap to next action |
| Adaptive exam feedback | IELTS and SAT Mini Mock Studio |
| Natural-language schedule parsing | Editable weekly Smart Routine |
| Bengali structured generation | Bengali landing page and workspace |
| Model allowlist assertion | Gemma 4-only runtime boundary |

The latest public build also includes a responsive 416px Strategist rail, a compact full-page Strategist command deck, a functional browser-local profile and settings center, official-video learning, and live student-offer discovery.


## Responsible use and submission links

- Polaris provides planning support, not admission guarantees.
- IELTS and SAT questions are original, unofficial practice items.
- Evidence stays incomplete until a human can verify the artifact and outcome.
- If Gemma 4 is unavailable, the application labels the deterministic fallback and never switches to another language model.
- Public demo profile and settings changes stay in the browser and are separate from real accounts.

**Live application:** https://polaris-gemma4.vercel.app/  
**Judge workspace:** https://polaris-gemma4.vercel.app/demo  
**Source:** https://github.com/ImtiazHossain-Eshan/polaris-gemma4

Built by **Imtiaz Hossain** and **Mofftasim Hossain Sayem**.
